# Insurance Claim Fraud & Risk Prediction - Demonstration Notebook
Input data -> processing -> model -> fraud predictions, risk alerts, patterns.

Run from the project root (`jupyter notebook`) after `pip install -r requirements.txt`.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from src.data_generation import generate_claims
from src.preprocessing import clean, engineer_features, TARGET
from src.modeling import (FEATURE_COLS, build_models, cross_validate_threshold,
                          evaluate, build_alerts)
from src.pattern_mining import mine_fraud_patterns

## 1. Input data (documented synthetic dataset)

In [ ]:
raw = generate_claims()
print(raw.shape)
raw.head()

In [ ]:
raw.isna().sum()[raw.isna().sum() > 0]

## 2. Processing: cleaning + feature engineering

In [ ]:
df = engineer_features(clean(raw))
print(df.shape, "fraud rate: %.2f%%" % (100 * df[TARGET].mean()))
df[["claim_id","amount_to_coverage","new_customer","late_report","text_flag_count",TARGET]].head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
df[TARGET].value_counts().plot.bar(ax=ax[0], title="Class distribution (0=legit, 1=fraud)")
df.groupby("claim_type")[TARGET].mean().plot.barh(ax=ax[1], title="Fraud rate by claim type")
plt.tight_layout(); plt.show()

## 3. Model training and evaluation (vs. baseline)

In [ ]:
X, y = df[FEATURE_COLS], df[TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
models, rows = build_models(), []
for name, m in models.items():
    m.fit(X_tr, y_tr)
    thr = 0.5 if name.startswith("Baseline") else cross_validate_threshold(m, X_tr, y_tr)
    rows.append(evaluate(name, m, X_te, y_te, thr))
results = pd.DataFrame(rows).set_index("model")
results[["threshold","roc_auc","pr_auc","precision","recall","f1"]].round(3)

## 4. Fraud predictions and risk alerts

In [ ]:
best = results.drop(index="Baseline (majority class)")["pr_auc"].idxmax()
thr = float(results.loc[best, "threshold"])
alerts = build_alerts(df.loc[X_te.index], models[best].predict_proba(X_te)[:, 1], thr)
print("Best model:", best, "| threshold:", round(thr, 2))
print(alerts["risk_level"].value_counts())
alerts.head(10)

## 5. Recurring fraud patterns (Apriori, ranked by lift)

In [ ]:
patterns = mine_fraud_patterns(df)
patterns[["pattern","support_in_fraud","fraud_rate_when_present","lift"]]

In [ ]:
d = patterns.head(10).iloc[::-1]
plt.figure(figsize=(9, 4)); plt.barh(d["pattern"], d["lift"]); plt.axvline(1, color="k", ls="--")
plt.title("Top fraud patterns by lift"); plt.tight_layout(); plt.show()

## 6. Score a new claim

In [ ]:
from src.predict import score_claim
bundle = {"model": models[best], "threshold": thr, "name": best}
score_claim(bundle, dict(customer_age=29, policy_tenure_months=4, previous_claims=3, claim_type="theft",
    region="East", policy_coverage=100000, claim_amount=82000, report_delay_days=21,
    police_report_filed="No", num_witnesses=0, coverage_changed_recently="Yes", incident_hour=2,
    claim_description="Stolen overnight, no cctv, urgent settlement needed. Requesting cash settlement immediately."))